In [3]:
import os
import csv
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
# import matplotlib.pyplot as plt
# from PIL import Image

In [4]:
csv_path = "train.csv"
rows = []
with open(csv_path, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        rows.append(row)

print(f"Wczytano {len(rows)} przykładów")

Wczytano 12672 przykładów


In [6]:
import cv2
import numpy as np

def lbp_hist(img_path, size=(64, 64)):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

    if img is None:
        raise ValueError("Nie można wczytać obrazu")

    img = cv2.resize(img, size)

    h, w = img.shape
    lbp = np.zeros((h, w), dtype=np.uint8)

    neighbors = [
        (-1, -1), (-1, 0), (-1, 1),
        (0, 1),
        (1, 1), (1, 0), (1, -1),
        (0, -1)
    ]

    for y in range(1, h - 1):
        for x in range(1, w - 1):
            center = img[y, x]
            code = 0

            for i, (dy, dx) in enumerate(neighbors):
                if img[y + dy, x + dx] >= center:
                    code |= (1 << i)

            lbp[y, x] = code

    hist = np.zeros(256, dtype=np.float32)

    for value in lbp.ravel():
        hist[value] += 1

    hist /= (hist.sum() + 1e-7)

    return hist

In [7]:
X = []
y = []
valid_paths = []
skipped = 0

for row in rows:
    img_path = row["image_path"]
    label = int(row["label"])
    
    if not os.path.exists(img_path):
        print(f"Pominięty nieistniejący: {img_path}")
        skipped += 1
        continue
    
    hist = lbp_hist(img_path)
    X.append(hist)
    y.append(label)
    valid_paths.append(img_path)

X = np.array(X)
y = np.array(y)

print(f"W przygotowaniu: {len(X)} przykładów")
print(f"Pominiono: {skipped}")
print(f"Wymiary X: {X.shape}")

W przygotowaniu: 12672 przykładów
Pominiono: 0
Wymiary X: (12672, 256)


In [8]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)
print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

Train: 8870, Val: 1901, Test: 1901


In [9]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

train_acc = rf.score(X_train, y_train)
val_acc = rf.score(X_val, y_val)
test_acc = rf.score(X_test, y_test)

print("\n" + "="*50)
print("WYNIKI AKURACJI")
print("="*50)
print(f"Akuracja train: {train_acc:.3f}")
print(f"Akuracja val:   {val_acc:.3f}")
print(f"Akuracja test:  {test_acc:.3f}")


WYNIKI AKURACJI
Akuracja train: 1.000
Akuracja val:   0.691
Akuracja test:  0.690


In [ ]:
y_pred = rf.predict(X_test)
print("\n" + "="*50)
print("CLASSIFICATION REPORT (test)")
print("="*50)
print(classification_report(y_test, y_pred))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion matrix:")
print(cm)

# ========================
# 7. Wizualizacja histogramów
# ========================
sample_labels = [0, 3, 6, 7, 9, 12]

name_map = {
    0: "empty",
    3: "black_rook",
    6: "black_pawn",
    7: "white_king",
    9: "white_rook",
    12: "white_pawn"
}

plt.figure(figsize=(10, 6))
for label in sample_labels:
    idxs = np.where(y == label)[0]
    if len(idxs) == 0:
        continue
    mean_hist = np.mean(X[idxs], axis=0)
    plt.plot([0, 1], mean_hist, marker="o", label=name_map[label])

plt.xlabel("Wartość binarna (0/1)")
plt.ylabel("Liczba pikseli")
plt.legend()
plt.title("Średni histogram binarny dla wybranych klas")
plt.grid(True)
plt.tight_layout()
plt.savefig("binary_histogram_avg.png", dpi=150)
plt.close()

In [ ]:
import joblib
joblib.dump(rf, "chess_rf_model.joblib")
print("Zapisano chess_rf_model.joblib")